In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("🔄 Step 1: Loading Datasets directly from your workspace...")
# Loading directly from the root directory since the files are right there
jd_df = pd.read_csv("job_descriptions.csv") 
candidates_df = pd.read_csv("candidate_profiles.csv")

print("🔄 Step 2: Compiling context vectors from your exact column headers...")
candidates_df['combined_profile'] = (
    "Title: " + candidates_df['current_title'].fillna('') + 
    " | Matched: " + candidates_df['matched_skills'].fillna('') + 
    " | Missing: " + candidates_df['missing_skills'].fillna('') + 
    " | Details: " + candidates_df['fit_explanation'].fillna('')
)

jd_df['combined_jd'] = (
    "Title: " + jd_df['title'].fillna('') + 
    " | Required: " + jd_df['required_skills'].fillna('') + 
    " | Preferred: " + jd_df['preferred_skills'].fillna('') + 
    " | Duties: " + jd_df['responsibilities'].fillna('')
)

print("🔄 Step 3: Initializing AI model and generating Embeddings...")
model = SentenceTransformer('all-MiniLM-L6-v2')

jd_embeddings = model.encode(jd_df['combined_jd'].tolist(), show_progress_bar=True)
candidate_embeddings = model.encode(candidates_df['combined_profile'].tolist(), show_progress_bar=True)

print("🔄 Step 4: Computing contextual alignment scores...")
similarity_matrix = cosine_similarity(jd_embeddings, candidate_embeddings)

print("🔄 Step 5: Formatting results into the final ranking file...")
final_rankings = []
for jd_index, jd_row in jd_df.iterrows():
    scores = similarity_matrix[jd_index]
    for candidate_index, score in enumerate(scores):
        final_rankings.append({
            "job_id": jd_row['jd_id'],
            "candidate_id": candidates_df.iloc[candidate_index]['candidate_id'],
            "score": score
        })

submission_df = pd.DataFrame(final_rankings)
submission_df = submission_df.sort_values(by=["job_id", "score"], ascending=[True, False])

# Export the submission file to your root workspace
submission_df.to_csv("ranking_submission.csv", index=False)
print("✅ Done! 'ranking_submission.csv' has been generated in your workspace folder.")

C:\Users\Darshan Baisane\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔄 Step 1: Loading Datasets directly from your workspace...
🔄 Step 2: Compiling context vectors from your exact column headers...
🔄 Step 3: Initializing AI model and generating Embeddings...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.06it/s]


🔄 Step 4: Computing contextual alignment scores...
🔄 Step 5: Formatting results into the final ranking file...
✅ Done! 'ranking_submission.csv' has been generated in your workspace folder.
